# Multi-Model Prediction Visualization
이 노트북은 여러 모델의 예측 결과를 하나의 plot에 시각화합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
import seaborn as sns

# 한글 폰트 설정 (선택사항)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 파스텔톤 색상 팔레트
PASTEL_COLORS = [
    '#FFB3BA',  # 파스텔 핑크
    '#BAFFC9',  # 파스텔 그린
    '#BAE1FF',  # 파스텔 블루
    '#FFFFBA',  # 파스텔 옐로우
    '#FFD9BA',  # 파스텔 오렌지
    '#E0BBE4',  # 파스텔 퍼플
    '#FFDFD3',  # 파스텔 피치
    '#C7CEEA',  # 파스텔 라벤더
    '#B5EAD7',  # 파스텔 민트
    '#FFE5B4',  # 파스텔 베이지
]

## 설정: 여기서 경로와 모델들을 지정하세요

In [ ]:
# ==================== 사용자 설정 ====================

# 기본 경로 (jupyter 폴더에서 실행할 경우 ../saved_model)
BASE_DIR = "../saved_model"

# Data version
DATA_VERSION = "version2_250813_processed"

# Environment name
ENVIRONMENT_NAME = "livingroom"  # 또는 "room1", "room2" 등

# 비교할 모델들과 버전 정보
# 형식: {"모델명": "버전폴더명"}
MODELS_TO_PLOT = {
    "MICN": "version0",
    "PatchTST": "version0",
    # 필요한 만큼 추가
    # "ModelName": "version1",
}

# 시스템 제어 타입
CONTROL_TYPE = "system_control"

# Plot 설정
FIGSIZE = (20, 8)  # 그림 크기 (n_rows=1일 때)
SAMPLE_RANGE = None  # None이면 전체, 또는 (start, end) 튜플
SHOW_LEGEND = True
ALPHA = 0.7  # 투명도 (0~1)
N_ROWS = 4  # 데이터를 몇 개의 행으로 나눌지 (1이면 한 줄에 모두 표시)
ROW_HEIGHT = 4  # 각 행의 높이

# =====================================================

## 데이터 로딩 함수

In [ ]:
def load_predictions(base_dir, data_version, environment, model_name, control_type, version):
    """
    특정 모델의 예측값과 실제값을 로드합니다.
    경로: base_dir/data_version/environment/model_name/control_type/version
    
    Returns:
        outputs (np.ndarray): 예측값
        targets (np.ndarray): 실제값
    """
    model_dir = Path(base_dir) / data_version / environment / model_name / control_type / version
    
    outputs_path = model_dir / "TEST_total_outputs.npy"
    targets_path = model_dir / "TEST_total_targets.npy"
    
    if not outputs_path.exists():
        raise FileNotFoundError(f"Outputs file not found: {outputs_path}")
    if not targets_path.exists():
        raise FileNotFoundError(f"Targets file not found: {targets_path}")
    
    outputs = np.load(outputs_path)
    targets = np.load(targets_path)
    
    print(f"✅ Loaded {model_name}/{version}:")
    print(f"   Path: {model_dir}")
    print(f"   Outputs shape: {outputs.shape}")
    print(f"   Targets shape: {targets.shape}")
    
    return outputs, targets


def load_all_models(base_dir, data_version, environment, models_dict, control_type):
    """
    모든 모델의 데이터를 로드합니다.
    
    Returns:
        data_dict: {model_name: {"outputs": array, "targets": array}}
    """
    data_dict = {}
    
    for model_name, version in models_dict.items():
        try:
            outputs, targets = load_predictions(
                base_dir, data_version, environment, model_name, control_type, version
            )
            data_dict[model_name] = {
                "outputs": outputs,
                "targets": targets,
                "version": version
            }
        except Exception as e:
            print(f"❌ Error loading {model_name}: {e}")
            continue
    
    return data_dict

## 데이터 로드

In [ ]:
# 모든 모델의 데이터 로드
all_data = load_all_models(
    BASE_DIR,
    DATA_VERSION,
    ENVIRONMENT_NAME,
    MODELS_TO_PLOT,
    CONTROL_TYPE
)

print(f"\n📊 Successfully loaded {len(all_data)} models")

## 시각화

In [ ]:
def flatten_data(arr):
    """배열을 1D로 flatten합니다."""
    if arr.ndim == 3:
        return arr.squeeze(-1).flatten()
    elif arr.ndim == 2:
        return arr.flatten()
    return arr


def plot_predictions(data_dict, sample_range=None, figsize=(20, 8), alpha=0.7, 
                     show_legend=True, n_rows=1, row_height=5):
    """
    모든 모델의 예측 결과를 plot합니다.
    
    Args:
        data_dict: load_all_models의 출력
        sample_range: None 또는 (start, end) 튜플
        figsize: 그림 크기 (n_rows=1일 때만 사용)
        alpha: 선 투명도
        show_legend: 범례 표시 여부
        n_rows: 행 개수 (데이터를 n_rows개로 나눠서 표시)
        row_height: 각 행의 높이
    """
    if not data_dict:
        print("❌ No data to plot!")
        return
    
    # 첫 번째 모델의 targets를 가져옴
    first_model = list(data_dict.keys())[0]
    targets = flatten_data(data_dict[first_model]["targets"])
    print(f"📝 Targets shape: {targets.shape}")
    
    # Sample range 설정
    if sample_range is None:
        start, end = 0, len(targets)
    else:
        start, end = sample_range
    
    total_length = end - start
    points_per_row = total_length // n_rows
    
    # Figure 생성
    if n_rows == 1:
        fig, axes = plt.subplots(1, 1, figsize=figsize)
        axes = [axes]
    else:
        fig, axes = plt.subplots(n_rows, 1, figsize=(figsize[0], row_height * n_rows))
    
    # 각 행에 대해 플롯
    for row_idx in range(n_rows):
        ax = axes[row_idx]
        
        # 현재 행의 데이터 범위
        row_start = start + row_idx * points_per_row
        if row_idx == n_rows - 1:
            row_end = end
        else:
            row_end = start + (row_idx + 1) * points_per_row
        
        x = np.arange(row_start, row_end)
        targets_subset = targets[row_start:row_end]
        
        # Ground Truth 그리기
        ax.plot(x, targets_subset, 
                color='black', 
                linewidth=2, 
                label='Ground Truth' if row_idx == 0 else None,
                zorder=100)
        
        # 각 모델의 예측값 그리기
        for idx, (model_name, data) in enumerate(data_dict.items()):
            outputs = flatten_data(data["outputs"])
            outputs_subset = outputs[row_start:row_end]
            version = data["version"]
            color = PASTEL_COLORS[idx % len(PASTEL_COLORS)]
            
            ax.plot(x, outputs_subset,
                    color=color,
                    linewidth=1.5,
                    label=f'{model_name} ({version})' if row_idx == 0 else None,
                    alpha=alpha)
        
        # 각 subplot 꾸미기
        ax.set_ylabel('Value', fontsize=12)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.set_facecolor('#F8F9FA')
        ax.set_title(f'Time Steps: {row_start} ~ {row_end-1}', fontsize=11, loc='right')
    
    # 마지막 subplot에 x label
    axes[-1].set_xlabel('Time Steps', fontsize=14, fontweight='bold')
    
    # 전체 제목
    fig.suptitle(f'Model Predictions Comparison\n{DATA_VERSION} - {ENVIRONMENT_NAME}', 
                 fontsize=16, fontweight='bold', y=1.02)
    
    # 범례
    if show_legend:
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc='upper right', 
                #    bbox_to_anchor=(0.5, 1.0), ncol=min(5, len(labels)),
                   fontsize=12)
        # fig.legend(handles, labels, loc='upper center', 
        #            bbox_to_anchor=(0.5, 1.0), ncol=min(5, len(labels)),
        #            fontsize=10, framealpha=0.9)
    plt.tight_layout()
    plt.show()
    
    # 통계 정보 출력
    targets_full = targets[start:end]
    print("\n📈 Statistics:")
    print(f"Total time steps: {total_length} (split into {n_rows} rows)")
    print(f"\nGround Truth - Mean: {targets_full.mean():.4f}, Std: {targets_full.std():.4f}")
    print("\nModel Predictions:")
    for model_name, data in data_dict.items():
        outputs = flatten_data(data["outputs"])[start:end]
        mae = np.mean(np.abs(outputs - targets_full))
        rmse = np.sqrt(np.mean((outputs - targets_full)**2))
        print(f"  {model_name:15s} - MAE: {mae:8.4f}, RMSE: {rmse:8.4f}")

In [ ]:
# Plot 생성
plot_predictions(
    all_data,
    sample_range=SAMPLE_RANGE,
    figsize=FIGSIZE,
    alpha=ALPHA,
    show_legend=SHOW_LEGEND,
    n_rows=8,
    row_height=ROW_HEIGHT
)

## (Optional) 특정 구간만 확대해서 보기

In [ ]:
# 예시: 0~500 구간만 보기 (1행으로)
plot_predictions(
    all_data,
    sample_range=(380520, 507359),
    figsize=(30, 6),
    alpha=0.8,
    show_legend=True,
    n_rows=2  # 짧은 구간은 1행으로
)

## (Optional) 여러 환경을 비교하기

In [ ]:
def plot_multiple_environments(base_dir, data_version, environments, model_name, control_type, version):
    """
    하나의 모델에 대해 여러 환경의 결과를 비교합니다.
    """
    fig, axes = plt.subplots(len(environments), 1, figsize=(20, 6 * len(environments)))
    
    if len(environments) == 1:
        axes = [axes]
    
    for idx, env in enumerate(environments):
        try:
            outputs, targets = load_predictions(
                base_dir, data_version, env, model_name, control_type, version
            )
            
            # Flatten if 3D
            if targets.ndim == 3:
                targets = targets.squeeze(-1).flatten()
                outputs = outputs.squeeze(-1).flatten()
            elif targets.ndim == 2:
                targets = targets.flatten()
                outputs = outputs.flatten()
            
            x = np.arange(len(targets))
            
            axes[idx].plot(x, targets, color='black', linewidth=2.5, label='Ground Truth', zorder=100)
            axes[idx].plot(x, outputs, color=PASTEL_COLORS[idx % len(PASTEL_COLORS)], 
                          linewidth=2, label=f'{model_name} Prediction', alpha=0.7)
            
            axes[idx].set_title(f'{env}', fontsize=14, fontweight='bold')
            axes[idx].set_xlabel('Time Steps', fontsize=12)
            axes[idx].set_ylabel('Value', fontsize=12)
            axes[idx].grid(True, alpha=0.3, linestyle='--')
            axes[idx].set_facecolor('#F8F9FA')
            axes[idx].legend(loc='best', fontsize=10)
            
        except Exception as e:
            print(f"❌ Error loading {env}: {e}")
            continue
    
    plt.tight_layout()
    plt.show()

# 사용 예시:
# plot_multiple_environments(
#     BASE_DIR,
#     DATA_VERSION,
#     ["livingroom", "room1", "room2"],
#     "PatchTST",
#     CONTROL_TYPE,
#     "version0"
# )

## (Optional) 저장하기

In [ ]:
def save_plot(data_dict, save_path, sample_range=None, figsize=(20, 8), 
              alpha=0.7, dpi=300, n_rows=1, row_height=5):
    """
    Plot을 파일로 저장합니다.
    """
    if not data_dict:
        print("❌ No data to plot!")
        return
    
    first_model = list(data_dict.keys())[0]
    targets = flatten_data(data_dict[first_model]["targets"])
    
    if sample_range is None:
        start, end = 0, len(targets)
    else:
        start, end = sample_range
    
    total_length = end - start
    points_per_row = total_length // n_rows
    
    if n_rows == 1:
        fig, axes = plt.subplots(1, 1, figsize=figsize)
        axes = [axes]
    else:
        fig, axes = plt.subplots(n_rows, 1, figsize=(figsize[0], row_height * n_rows))
    
    for row_idx in range(n_rows):
        ax = axes[row_idx]
        
        row_start = start + row_idx * points_per_row
        row_end = end if row_idx == n_rows - 1 else start + (row_idx + 1) * points_per_row
        
        x = np.arange(row_start, row_end)
        targets_subset = targets[row_start:row_end]
        
        ax.plot(x, targets_subset, color='black', linewidth=2, 
                label='Ground Truth' if row_idx == 0 else None, zorder=100)
        
        for idx, (model_name, data) in enumerate(data_dict.items()):
            outputs = flatten_data(data["outputs"])
            outputs_subset = outputs[row_start:row_end]
            version = data["version"]
            color = PASTEL_COLORS[idx % len(PASTEL_COLORS)]
            
            ax.plot(x, outputs_subset, color=color, linewidth=1.5, 
                   label=f'{model_name} ({version})' if row_idx == 0 else None, alpha=alpha)
        
        ax.set_ylabel('Value', fontsize=12)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.set_facecolor('#F8F9FA')
        ax.set_title(f'Time Steps: {row_start} ~ {row_end-1}', fontsize=11, loc='right')
    
    axes[-1].set_xlabel('Time Steps', fontsize=14, fontweight='bold')
    fig.suptitle(f'Model Predictions Comparison\n{DATA_VERSION} - {ENVIRONMENT_NAME}', 
                 fontsize=16, fontweight='bold', y=1.02)
    
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', 
               bbox_to_anchor=(0.5, 1.0), ncol=min(5, len(labels)),
               fontsize=10, framealpha=0.9)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=dpi, bbox_inches='tight')
    plt.close()
    
    print(f"✅ Plot saved to: {save_path}")

# 사용 예시:
# save_plot(all_data, "model_comparison.png", n_rows=4, dpi=300)